# zynarai v1 — ACE-Step 5Hz LM Planner LoRA (Colab Pro)

Fine-tune the **5Hz LM planner** (Qwen3 **0.6B**) to transfer composition/groove
character from the 57-track core — the half the DiT LoRA can't touch. The DiT LoRA
gave you the *sound* (sub-bass, production); this gives you the *arrangement*.

Pipeline: load base model → extract `<|audio_code_N|>` targets from ~30s chunks →
SFT the Qwen3 LM with a LoRA to predict codes from the caption.

**Two honest caveats baked into the config:** (1) 57 tracks is thin for LM SFT, so we
chunk to ~30s (~350-450 examples) and use the smallest LM; (2) the prompt format must
match inference exactly — a round-trip check cell verifies this before full training.

In [ ]:
# 1 — GPU + deps. Reuses the ACE-Step env; adds PEFT/TRL for standard causal-LM SFT.
!nvidia-smi --query-gpu=name,memory.total --format=csv
%cd /content
![ -d ACE-Step-1.5 ] || git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
lines = open('requirements.txt').read().splitlines()
SKIP = ('torch==','torchvision==','torchaudio==','flash-attn','--extra-index-url','triton')
open('/tmp/req.txt','w').write('\n'.join(l for l in lines if not any(t in l for t in SKIP)))
!pip install -q -r /tmp/req.txt
!pip install -q loguru lycoris-lora vector_quantize_pytorch
!pip install -q 'peft>=0.18' 'datasets>=2.19'
print('deps ready')

In [ ]:
# 1b — meta-tensor patch (same as the DiT notebook; base model loads the audio tokenizer)
import os, re, vector_quantize_pytorch
vq = os.path.dirname(vector_quantize_pytorch.__file__)
for fn in os.listdir(vq):
    if fn.endswith('.py'):
        p = os.path.join(vq, fn); t = open(p).read()
        t2 = re.sub(r'assert \(([^)]+?) > 1\)\.all\(\)',
                    r"assert \1.device.type == 'meta' or (\1 > 1).all()", t)
        if t2 != t: open(p,'w').write(t2); print('patched', fn)
print('vq patched')

In [ ]:
# 2 — Drive + paths (same symlink trick for the space in 'lora fine tune')
from google.colab import drive; drive.mount('/content/drive')
import os
from pathlib import Path
if not os.path.exists('/content/loradrive'):
    os.symlink('/content/drive/MyDrive/lora fine tune', '/content/loradrive')
DRIVE   = Path('/content/loradrive')
DATASET = DRIVE / 'dataset'
SFT_JSONL = DRIVE / 'lm_sft' / 'sft.jsonl'          # extracted training data (cached)
LM_OUT    = DRIVE / 'output' / 'lm_lora_0.6b'       # LM adapter checkpoints
for p in (SFT_JSONL.parent, LM_OUT): p.mkdir(parents=True, exist_ok=True)
assert (DATASET/'dataset.json').exists(), f'upload dataset to {DATASET}'
print('dataset:', DATASET, '| examples cache:', SFT_JSONL)

In [ ]:
# 3 — Checkpoints + copy the audio + our prep helpers
CKPT='/content/checkpoints'
from huggingface_hub import snapshot_download
snapshot_download('ACE-Step/Ace-Step1.5', local_dir=CKPT)
snapshot_download('ACE-Step/acestep-v15-base', local_dir=f'{CKPT}/acestep-v15-base')
# UI/handler scans <repo>/checkpoints — symlink so model discovery works
dst='/content/ACE-Step-1.5/checkpoints'
if not os.path.islink(dst) and not os.path.exists(dst): os.symlink(CKPT, dst)
import shutil; shutil.copytree(str(DATASET), '/content/data', dirs_exist_ok=True)
# our prep module (from the electronic-lora repo — upload scripts/lm_sft_prep.py to Drive first)
shutil.copy(str(DRIVE/'scripts'/'lm_sft_prep.py'), '/content/lm_sft_prep.py')
print('audio files:', len(list(Path('/content/data/audio').glob('*'))))

In [ ]:
# 4 — Load the base model (gives us the audio->codes encoder)
import sys; sys.path.insert(0, '/content/ACE-Step-1.5')
from acestep.handler import AceStepHandler
base_handler = AceStepHandler()
# config_path = the base model dir; matches what the UI loads as 'base'
status, ok = base_handler.initialize_service(
    project_root='/content/ACE-Step-1.5',
    config_path=f'{CKPT}/acestep-v15-base',
    device='cuda',
    compile_model=False,
)
print(status); assert ok, 'model failed to load'

In [ ]:
# 5 — Round-trip check on ONE chunk BEFORE full extraction.
#     Verifies the audio->codes encoder + the 5Hz rate. (The chat-template FORMAT
#     is verified in cell 8, after the LM tokenizer loads.)
import importlib, lm_sft_prep as P; importlib.reload(P)
import json, tempfile
ds = json.load(open('/content/data/dataset.json'))
sample = ds['samples'][0]
track = f"/content/data/audio/{sample['filename']}"
idx, samples, sr, dur = next(P.iter_chunks(track))
wav = P.write_chunk_wav(samples, sr, tempfile.mktemp(suffix='.wav'))
codes = base_handler.convert_src_audio_to_codes(wav)
print('ERROR' if P.is_error_codes(codes) else 'OK')
n = codes.count('<|audio_code_')
print(f'n code tokens: {n}  (expect ~5*sec = {round(5*dur)})')
print('codes preview:', codes[:120])

In [ ]:
# 6 — Full extraction: 57 tracks -> ~30s chunks -> RAW sft.jsonl (cached on Drive).
#     Stores {caption, lyrics, codes}; the chat prompt is built later with the LM tokenizer.
import json, tempfile, importlib, lm_sft_prep as P; importlib.reload(P)
from pathlib import Path
if SFT_JSONL.exists() and SFT_JSONL.stat().st_size > 0:
    n = sum(1 for _ in open(SFT_JSONL)); print(f'cache hit: {n} examples — skipping extraction')
else:
    ds = json.load(open('/content/data/dataset.json'))
    records = []
    for s in ds['samples']:
        track = f"/content/data/audio/{s['filename']}"
        if not Path(track).exists(): print('missing:', s['filename']); continue
        k = 0
        for idx, samples, sr, dur in P.iter_chunks(track, chunk_s=30.0, min_s=8.0):
            wav = P.write_chunk_wav(samples, sr, tempfile.mktemp(suffix='.wav'))
            codes = base_handler.convert_src_audio_to_codes(wav); Path(wav).unlink(missing_ok=True)
            if P.is_error_codes(codes): print('skip', s['filename'], idx, codes[:50]); continue
            records.append(P.build_record(s['caption'], codes, source=s['filename'], chunk_index=idx)); k += 1
        print(f"{s['filename'][:45]:45s} -> {k} chunks")
    n = P.write_jsonl(records, SFT_JSONL)
    print(f'wrote {n} raw SFT examples to {SFT_JSONL}')

In [ ]:
# 7 — Free DiT/VAE, download + load the Qwen3 0.6B 5Hz LM for SFT.
#     The 0.6B is a SEPARATE HF repo — the main snapshot only ships the 1.7B default,
#     so we fetch it explicitly into checkpoints/ (also makes it selectable in the UI).
import gc, os, torch
if 'base_handler' in globals():
    del base_handler
gc.collect(); torch.cuda.empty_cache()

from huggingface_hub import snapshot_download
LM_PATH = f'{CKPT}/acestep-5Hz-lm-0.6B'
if not os.path.isdir(LM_PATH) or not os.listdir(LM_PATH):
    snapshot_download('ACE-Step/acestep-5Hz-lm-0.6B', local_dir=LM_PATH)   # ~1.2GB
print('LM path:', LM_PATH)

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(LM_PATH, trust_remote_code=True, use_fast=True)
lm  = AutoModelForCausalLM.from_pretrained(LM_PATH, trust_remote_code=True,
                                           torch_dtype=torch.bfloat16, device_map='cuda')
print('LM loaded:', lm.config.model_type, '| vocab:', len(tok))

In [ ]:
# 8 — Build SFT text with the REAL LM chat template (matches inference exactly).
#     Robust to old/new jsonl schema, and the FORMAT-VERIFY gate: eyeball one example.
import importlib, json, lm_sft_prep as P; importlib.reload(P)
from datasets import Dataset

# caption lookup by filename, for jsonl written before the caption/codes schema fix
ds_meta = json.load(open('/content/data/dataset.json'))
cap_by_file = {s['filename']: s['caption'] for s in ds_meta['samples']}

rows = []
with open(SFT_JSONL) as f:
    for line in f:
        r = json.loads(line)
        codes   = r.get('codes') or r.get('completion')                    # new | old key
        caption = r.get('caption') or cap_by_file.get(r.get('source',''), '')
        if not codes or not caption:
            continue
        rows.append({'text': P.build_lm_text(tok, caption, codes,
                                              lyrics=r.get('lyrics', '[Instrumental]'))})
print(f'{len(rows)} examples rendered')

ds_all = Dataset.from_list(rows)
splits = ds_all.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = splits['train'], splits['test']
print('=== rendered example — VERIFY: system=Instruction, user=Caption+Lyric, then <think>+codes ===')
print(train_ds[0]['text'][:600])
print('   ...(tail)...')
print(train_ds[0]['text'][-160:])

In [ ]:
# 9 — LoRA SFT via HF Trainer + manual completion-only masking (version-robust, no TRL).
import torch, importlib, lm_sft_prep as P; importlib.reload(P)
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# completion begins right after the assistant-turn opener; mask everything up to it
RESP = tok('<|im_start|>assistant\n', add_special_tokens=False)['input_ids']

def find_sub(seq, sub):
    for i in range(len(seq) - len(sub), -1, -1):
        if seq[i:i+len(sub)] == sub:
            return i
    return -1

def encode(ex):
    ids = tok(ex['text'], add_special_tokens=False, truncation=True, max_length=1024)['input_ids']
    labels = list(ids)
    j = find_sub(ids, RESP)
    cut = (j + len(RESP)) if j != -1 else len(ids)
    labels[:cut] = [-100] * cut
    return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': labels}

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(encode,  remove_columns=eval_ds.column_names)

ex0 = train_tok[0]
n_train_tokens = sum(1 for x in ex0['labels'] if x != -100)
print(f"example 0: {len(ex0['input_ids'])} tokens, {n_train_tokens} trained (rest masked)")
assert n_train_tokens > 0, 'all tokens masked — response marker not found; check RESP'

def collate(batch):
    maxlen = max(len(b['input_ids']) for b in batch)
    pad = lambda x, v: x + [v]*(maxlen - len(x))
    return {
        'input_ids':      torch.tensor([pad(b['input_ids'], tok.pad_token_id) for b in batch]),
        'attention_mask': torch.tensor([pad(b['attention_mask'], 0) for b in batch]),
        'labels':         torch.tensor([pad(b['labels'], -100) for b in batch]),
    }

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj'])
lm_lora = get_peft_model(lm, peft_cfg)
lm_lora.print_trainable_parameters()

args = TrainingArguments(
    output_dir=str(LM_OUT),
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=3, learning_rate=1e-4, lr_scheduler_type='cosine', warmup_ratio=0.05,
    logging_steps=10, eval_strategy='epoch', save_strategy='epoch',
    bf16=True, report_to='none', seed=42,
)
trainer = Trainer(model=lm_lora, args=args, train_dataset=train_tok,
                  eval_dataset=eval_tok, data_collator=collate)
trainer.train()
lm_lora.save_pretrained(str(LM_OUT / 'final'))
print('saved LM LoRA ->', LM_OUT / 'final')

## Using it

At inference, load **both** adapters: the DiT LoRA (your sound) and this LM LoRA
(your grooves). The LM adapter plugs into the 5Hz planner via PEFT.

## Evaluate

1. Same-seed A/B: LM-LoRA on vs off (DiT LoRA fixed). Listen for arrangement/groove
   shift — fill patterns, phrase pacing, drum placement — not timbre.
2. Watch **eval loss** in cell 9: if it climbs while train loss falls, you're
   overfitting the 57 tracks — drop to 1-2 epochs or lower rank.
3. Compare against DiT-LoRA-only: does adding the LM adapter make the composition
   read as "your tracks" rather than just "your sound"? That's the whole hypothesis.